# Module 10: Data Visualization

**Duration:** 18 hours  
**ML Focus:** EDA Visualization for ML Datasets

Data visualization is essential for understanding data, detecting patterns, informing feature engineering, and communicating results. This lesson covers matplotlib's architecture and seaborn's high-level API.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

%matplotlib inline
sns.set_theme()
print('Libraries loaded successfully.')

## 1. Matplotlib Philosophy: Figure and Axes

Matplotlib has two APIs: pyplot (state-machine, MATLAB-like) and OO (explicit Figure and Axes control). The OO API is preferred for serious work.

In [ ]:
# Pyplot style (simple, good for quick exploration)
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = iris.target_names[iris.target]

plt.figure(figsize=(8, 5))
plt.plot(df_iris['sepal length (cm)'], label='Sepal Length')
plt.plot(df_iris['petal length (cm)'], label='Petal Length')
plt.title('Pyplot Style')
plt.xlabel('Sample Index')
plt.ylabel('Length (cm)')
plt.legend()
plt.show()

# OO API (explicit, recommended)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_iris['sepal length (cm)'], df_iris['sepal width (cm)'],
           c=iris.target, cmap='viridis', alpha=0.7)
ax.set_title('OO API: Sepal Length vs Width')
ax.set_xlabel('Sepal Length (cm)')
ax.set_ylabel('Sepal Width (cm)')
ax.grid(True, alpha=0.3)
plt.show()

## 2. Scatter Plots for Feature Relationships

Scatter plots reveal relationships, clusters, and outliers between pairs of features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple scatter
axes[0].scatter(df_iris['petal length (cm)'], df_iris['petal width (cm)'],
                c=iris.target, cmap='Set2', alpha=0.8, s=50)
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title('Petal: Length vs Width')

# Scatter with size encoding
scatter = axes[1].scatter(df_iris['sepal length (cm)'], df_iris['petal length (cm)'],
                          c=iris.target, s=df_iris['petal width (cm)'] * 50,
                          cmap='plasma', alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Sepal Length (cm)')
axes[1].set_ylabel('Petal Length (cm)')
axes[1].set_title('Size = Petal Width')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Species')

plt.tight_layout()
plt.show()

## 3. Histograms, Box Plots, and Outlier Detection

Histograms show distributions; box plots reveal outliers via the IQR method.

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Histograms
axes[0, 0].hist(titanic['Age'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(titanic['Fare'], bins=50, color='coral', edgecolor='white')
axes[0, 1].set_title('Fare Distribution')
axes[0, 1].set_xlabel('Fare')
axes[0, 1].set_ylabel('Count')

# Histogram with KDE
axes[0, 2].hist(titanic['Age'].dropna(), bins=30, density=True, alpha=0.7,
                color='steelblue', edgecolor='white', label='Histogram')
kde_ages = np.linspace(0, 80, 200)
from scipy.stats import gaussian_kde
age_clean = titanic['Age'].dropna()
kde = gaussian_kde(age_clean)
axes[0, 2].plot(kde_ages, kde(kde_ages), 'r-', linewidth=2, label='KDE')
axes[0, 2].set_title('Age with KDE Overlay')
axes[0, 2].legend()

# Box plots
titanic.boxplot(column='Age', by='Pclass', ax=axes[1, 0])
axes[1, 0].set_title('Age by Passenger Class')
axes[1, 0].set_xlabel('Pclass')

titanic.boxplot(column='Fare', by='Pclass', ax=axes[1, 1])
axes[1, 1].set_title('Fare by Passenger Class')
axes[1, 1].set_ylabel('Fare')

# Outlier detection demo
Q1 = titanic['Fare'].quantile(0.25)
Q3 = titanic['Fare'].quantile(0.75)
IQR = Q3 - Q1
outliers = titanic[(titanic['Fare'] < Q1 - 1.5*IQR) | (titanic['Fare'] > Q3 + 1.5*IQR)]
axes[1, 2].scatter(range(len(outliers)), outliers['Fare'], color='red', alpha=0.6)
axes[1, 2].axhline(y=Q3 + 1.5*IQR, color='gray', linestyle='--', label='Upper fence')
axes[1, 2].axhline(y=Q1 - 1.5*IQR, color='gray', linestyle='--', label='Lower fence')
axes[1, 2].set_title(f'Fare Outliers (IQR method): {len(outliers)} detected')
axes[1, 2].set_xlabel('Outlier Index')
axes[1, 2].set_ylabel('Fare')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 4. Heatmaps for Correlation Analysis

Correlation heatmaps are essential for detecting multicollinearity and feature-target relationships.

In [ ]:
# Compute correlation matrix for Titanic numeric features
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_matrix = titanic[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')

# Add labels
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_matrix.columns)

# Add correlation values as text
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        text_color = 'white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', color=text_color, fontsize=11)

plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title('Titanic Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  - Pclass and Fare: strong negative correlation (-0.55)')
print('  - Pclass and Survived: moderate negative (-0.34)')
print('  - Fare and Survived: weak positive (0.26)')

## 5. Seaborn: High-Level API for Statistical Visualization

Seaborn simplifies complex statistical visualizations with sensible defaults and built-in grouping.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Count plot
sns.countplot(data=titanic, x='Survived', ax=axes[0, 0])
axes[0, 0].set_title('Survival Count')

# Histplot with hue
sns.histplot(data=titanic, x='Age', hue='Survived', bins=30, kde=True,
             alpha=0.5, ax=axes[0, 1])
axes[0, 1].set_title('Age Distribution by Survival')

# Box plot
sns.boxplot(data=titanic, x='Pclass', y='Fare', hue='Survived', ax=axes[0, 2])
axes[0, 2].set_title('Fare by Class and Survival')

# Bar plot
sns.barplot(data=titanic, x='Pclass', y='Survived', ax=axes[1, 0])
axes[1, 0].set_title('Survival Rate by Class')

# Violin plot
sns.violinplot(data=titanic, x='Pclass', y='Age', hue='Survived',
               split=True, ax=axes[1, 1])
axes[1, 1].set_title('Age Distribution by Class and Survival')

# Heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=axes[1, 2])
axes[1, 2].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

## 6. Advanced Seaborn: Pairplot, lmplot, and catplot

Seaborn's high-level functions create multi-panel visualizations with a single call.

In [ ]:
print('=== Pairplot: pairwise feature relationships ===')
sns.pairplot(df_iris, hue='species', diag_kind='kde', palette='Set2')
plt.suptitle('Iris Dataset: Pairplot', y=1.02)
plt.show()

print('\n=== lmplot: linear regression with confidence bands ===')
sns.lmplot(data=titanic, x='Age', y='Fare', hue='Survived',
           scatter_kws={'alpha': 0.4}, height=6)
plt.suptitle('Fare vs Age by Survival', y=1.02)
plt.show()

print('\n=== catplot: categorical multi-panel plots ===')
sns.catplot(data=titanic, x='Pclass', y='Age', hue='Survived',
            col='Sex', kind='box', height=5)
plt.suptitle('Age by Class, Sex, and Survival', y=1.05)
plt.show()

## 7. Customization: Colors, Labels, Legends, Annotations

Professional visualizations require attention to detail in customization.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Create a well-customized plot
survived = titanic[titanic['Survived'] == 1]
not_survived = titanic[titanic['Survived'] == 0]

ax.scatter(not_survived['Age'], not_survived['Fare'],
           c='#E74C3C', alpha=0.4, s=30, label='Not Survived', edgecolors='none')
ax.scatter(survived['Age'], survived['Fare'],
           c='#2ECC71', alpha=0.6, s=50, label='Survived', edgecolors='black', linewidth=0.5)

# Customization
ax.set_xlabel('Age (years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Fare ($)', fontsize=13, fontweight='bold')
ax.set_title('Titanic: Age vs Fare by Survival Status', fontsize=15, fontweight='bold', pad=15)
ax.legend(loc='upper right', frameon=True, shadow=True, fontsize=11)
ax.grid(True, alpha=0.3, linestyle=':')

# Annotations
ax.annotate('High-fare survivors', xy=(40, 250), xytext=(50, 280),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            fontsize=10, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

# Reference line
ax.axhline(y=titanic['Fare'].median(), color='gray', linestyle='--', alpha=0.5,
           label=f'Median Fare (${titanic["Fare"].median():.0f})')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Subplots with GridSpec

Complex multi-panel layouts require GridSpec for precise control.

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(15, 10))
gs = GridSpec(3, 3, figure=fig, width_ratios=[2, 1, 1], height_ratios=[1, 1, 1],
              hspace=0.3, wspace=0.3)

# Main scatter (spans larger area)
ax_main = fig.add_subplot(gs[:, 0])
ax_main.scatter(titanic['Age'], titanic['Fare'], c=titanic['Survived'],
                cmap='RdYlGn', alpha=0.5, edgecolors='gray', linewidth=0.3)
ax_main.set_xlabel('Age')
ax_main.set_ylabel('Fare')
ax_main.set_title('Age vs Fare (colored by Survival)')

# Top row
ax_age = fig.add_subplot(gs[0, 1])
ax_age.hist(titanic['Age'].dropna(), bins=20, color='steelblue', edgecolor='white')
ax_age.set_title('Age Distribution')

ax_fare = fig.add_subplot(gs[0, 2])
ax_fare.hist(titanic['Fare'], bins=30, color='coral', edgecolor='white')
ax_fare.set_title('Fare Distribution')

# Bottom row
ax_class = fig.add_subplot(gs[1, 1])
titanic['Pclass'].value_counts().sort_index().plot(kind='bar', ax=ax_class, color='green')
ax_class.set_title('Passenger Class')
ax_class.set_xlabel('Class')

ax_surv = fig.add_subplot(gs[1, 2])
titanic['Survived'].value_counts().plot(kind='bar', ax=ax_surv, color=['red', 'green'])
ax_surv.set_title('Survival')
ax_surv.set_xticklabels(['No', 'Yes'])

# Bottom row
ax_emb = fig.add_subplot(gs[2, 1])
titanic['Embarked'].value_counts().plot(kind='bar', ax=ax_emb, color='purple')
ax_emb.set_title('Embarkation Port')

ax_sex = fig.add_subplot(gs[2, 2])
titanic['Sex'].value_counts().plot(kind='bar', ax=ax_sex, color=['pink', 'blue'])
ax_sex.set_title('Sex')

plt.suptitle('Titanic Dataset Overview', fontsize=16, fontweight='bold', y=0.98)
plt.show()

## 9. Joint Plot and Boxen Plot

Joint plots show bivariate distributions with marginal histograms. Boxen plots (letter-value plots) handle large data better than traditional box plots.

In [ ]:
print('=== Joint Plot: bivariate distribution with marginals ===')
sns.jointplot(data=titanic, x='Age', y='Fare', kind='hex', height=8,
              marginal_kws=dict(bins=30, color='steelblue'))
plt.suptitle('Age vs Fare: Hexbin Joint Plot', y=1.02)
plt.show()

print('\n=== Boxen Plot: more detail than box plots ===')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxenplot(data=titanic, x='Pclass', y='Fare', ax=axes[0])
axes[0].set_title('Boxen Plot: Fare by Class')

sns.boxplot(data=titanic, x='Pclass', y='Fare', ax=axes[1])
axes[1].set_title('Box Plot: Fare by Class')
plt.tight_layout()
plt.show()

print('\nNote: Boxen plot shows more quantile levels for better distribution insight.')

## 10. Putting It All Together: ML-Focused EDA Report

A complete EDA workflow for the Iris dataset, documenting ML implications.

In [ ]:
print('=== Iris Dataset: Complete EDA Summary ===')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Pairwise scatter (selected)
sns.scatterplot(data=df_iris, x='petal length (cm)', y='petal width (cm)',
                hue='species', style='species', s=80, ax=axes[0, 0])
axes[0, 0].set_title('Petal: Length vs Width\n(>Strong separator)')

# 2. Sepal features
sns.scatterplot(data=df_iris, x='sepal length (cm)', y='sepal width (cm)',
                hue='species', style='species', s=80, ax=axes[0, 1])
axes[0, 1].set_title('Sepal: Length vs Width\n(>Some overlap)')

# 3. Feature distributions
for idx, (col, ax) in enumerate(zip(['sepal length (cm)', 'sepal width (cm)',
                                      'petal length (cm)', 'petal width (cm)'],
                                     [axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]])):
    for species in df_iris['species'].unique():
        subset = df_iris[df_iris['species'] == species]
        ax.hist(subset[col], bins=10, alpha=0.5, label=species[:4])
    ax.set_title(f'{col}\n(split by species)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nML Implications:')
print('  1. Petal features are highly discriminative -> strong predictors')
print('  2. Setosa is linearly separable from versicolor/virginica')
print('  3. Versicolor and virginica overlap in sepal dimensions')
print('  4. Petal length + petal width alone may be sufficient for many models')
print('\nModule 10 complete! You now have a full visualization toolkit.')